Install Yolov8 and Computer Vision

Importing Libraries


In [6]:
import cv2
import pandas as pd
from collections import defaultdict
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import os
import time

Upload Video

In [7]:
video_path = "Retail2.mp4"
print(f"Video Loaded: {video_path}")

Video Loaded: Retail2.mp4


Load Model

In [8]:
model = YOLO("yolov8n.pt") # YOLO

# DeepSORT
tracker = DeepSort(max_age=30, n_init=2, max_cosine_distance=0.4)

c:\Users\Noesis\real-time-video-analytics-retail\.venv\Lib\site-packages\deep_sort_realtime\embedder\embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Phase 1: Person Detection

In [9]:
# Frame counter
frame_number= 0

# Tracking
tracking_data = []
track_history = defaultdict(list)

# Customer Analytics
unique_customers = set()
max_simultaneous_customers = 0

# Zone Analytics
customer_zone = {}
zone_time = defaultdict(int)
zone_visit_count = defaultdict(int)

# Video Information
fps = 0
width = 0
height = 0

In [10]:
# Store Zones

Zone_A = (0, 0, 1280, 2160)
Zone_B = (1280, 0, 2560, 2160)
CHECKOUT = (2560, 0, 3840, 2160)

def get_zone(x,y):
  if x < 1280:
    return "Zone A"
  elif x < 2560:
    return "Zone B"
  else:
    return "Checkout"

In [11]:
# Video Setup
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_video = "retail_analytics_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

print(f"Resolution : {width} x {height}")
print(f"FPS : {fps}")

Resolution : 3840 x 2160
FPS : 30.0


In [12]:
# Video Processing (ENGINE)
while True:
  ret,frame = cap.read()
  if not ret:
    break

  frame_number +=1

  # Zone A
  cv2.rectangle(frame, (0,0), (1280,2160), (255,0,0), 3)
  cv2.putText(frame, "Zone A", (30,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)

  # Zone B
  cv2.rectangle(frame, (1280,0), (2560,2160), (0,255,255), 3)
  cv2.putText(frame, "Zone B", (1310,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

  # Checkout
  cv2.rectangle(frame, (2560,0), (3840,2160), (0,0,255), 3)
  cv2.putText(frame, "Checkout", (2590,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

  # YOLO Detection
  results = model(frame, verbose = False)
  detections = []

  # Person Detection
  for result in results:
    for box in result.boxes:
      class_id = int(box.cls[0])
      confidence = float(box.conf[0])

      if class_id == 0 and confidence >= 0.5:
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        width = x2 - x1
        height = y2 - y1
        detections.append(([x1,y1,width,height], confidence, "person"))

    # DeepSORT Tracking
    tracks = tracker.update_tracks(detections, frame=frame)
    active_tracks = 0

    # Customer Tracking
    for track in tracks:
      if not track.is_confirmed():
        continue
      active_tracks +=1
      track_id = track.track_id
      unique_customers.add(track_id)

      # Bounding Box
      left, top, right, bottom = map(int, track.to_ltrb())

      # Customer Center
      center_x = (left + right) // 2
      center_y = (top + bottom) // 2

      # Save Tracking Data
      tracking_data.append([frame_number, track_id, center_x, center_y])

      # Zone Analytics
      current_zone = get_zone(center_x, center_y)
      customer_zone[track_id] = current_zone
      zone_time[(track_id, current_zone)] += 1
      zone_visit_count[current_zone] += 1

      # Draw Bounding Box
      cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
      cv2.putText(frame, f"ID {track_id}", (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

      # Movement Trail
      track_history[track_id].append((center_x, center_y))

      if len(track_history[track_id]) > 30:
        track_history[track_id].pop(0)

      points = track_history[track_id]

      for i in range(1, len(points)):
        cv2.line(frame, points[i-1], points[i], (255, 0, 0), 2)

    # Frame Statistics
    max_simultaneous_customers = max(max_simultaneous_customers, active_tracks)
    cv2.putText(frame, f"Customers: {active_tracks}", (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()

print("Retail Analytics Processing Completed!")

Retail Analytics Processing Completed!


In [20]:
import shutil
print(shutil.which("ffmpeg"))

None


In [22]:
out.release()

import subprocess, os, imageio_ffmpeg

FFMPEG_PATH = imageio_ffmpeg.get_ffmpeg_exe()

raw_path = output_video
temp_path = "retail_analytics_output_temp.mp4"

subprocess.run([
    FFMPEG_PATH, "-y",
    "-i", raw_path,
    "-c:v", "libx264",
    "-profile:v", "high",
    "-level", "5.1",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    "-crf", "23",
    temp_path
], check=True)

os.replace(temp_path, raw_path)
print("Re-encoded for browser playback:", raw_path)

Re-encoded for browser playback: retail_analytics_output.mp4


In [13]:
print("Tracking records:", len(tracking_data))
print("Unique customers:", len(unique_customers))
print("Zone entries:", len(zone_time))

Tracking records: 278
Unique customers: 1
Zone entries: 2


Analytics Summary

In [14]:
# Customer Tracking CSV
tracking_df = pd.DataFrame(tracking_data, columns=["Frame","Customer ID", "Center_X", "Center_Y"])

tracking_df.to_csv("customer_tracking.csv",index=False)

# Dwell Time CSV
dwell_data = []
for (track_id, zone), frames in zone_time.items():
  dwell_data.append([track_id, zone,round (frames/ fps, 2)])

dwell_df = pd.DataFrame(dwell_data, columns=["Customer ID", "Zone", "Dwell Time (s)"])

dwell_df.to_csv("dwell_time.csv", index=False)

# Zone Analytics CSV
zone_df = pd.DataFrame(list(zone_visit_count.items()),columns=["Zone", "Visits"])

zone_df.to_csv("zone_analytics.csv", index=False)


# Project Summary
summary_df = pd.DataFrame({
    "Metric": [
        "Unique Customers",
        "Maximum Simultaneous Customers",
        "Total Tracking Records"
    ],
    "Value":[
        len(unique_customers),
        max_simultaneous_customers,
        len(tracking_data)
    ]
})

summary_df.to_csv("summary.csv", index=False)
print(summary_df)

                           Metric  Value
0                Unique Customers      1
1  Maximum Simultaneous Customers      1
2          Total Tracking Records    278


In [15]:
print(zone_df)
print(dwell_df)

       Zone  Visits
0  Checkout      29
1    Zone B     249
  Customer ID      Zone  Dwell Time (s)
0           1  Checkout            0.97
1           1    Zone B            8.30


Customer Heatmap

In [16]:
import numpy as np
max_x = max(row[2] for row in tracking_data) + 100
max_y = max(row[3] for row in tracking_data) + 100

heatmap = np.zeros((max_y, max_x), dtype=np.float32)

# Add Customer Locations
for row in tracking_data:
  x = int(row[2])
  y = int(row[3])

  if 0 <=x < heatmap.shape[1] and 0 <=y < heatmap.shape[0]:
    cv2.circle(heatmap, (x,y), 35, 1, -1) # Spreads each tracking point over an area

# Gaussian Blur
heatmap = cv2.GaussianBlur(heatmap,(201, 201), 0)

# Normalize Values
heatmap = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX)

heatmap = heatmap.astype(np.uint8)

# Color Map
colored_heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

# Save Heatmap
cv2.imwrite("customer_heatmap.png", colored_heatmap)


True

In [17]:
# Heatmap Overlay

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

if ret:
  # Resize heatmap to frame size
  colored_heatmap = cv2.resize(colored_heatmap, (frame.shape[1], frame.shape[0]))

  # Blend
  overlay = cv2.addWeighted(frame, 0.75, colored_heatmap, 0.25, 0)

  # Save
  cv2.imwrite("retail_heatmap_overlay.png", overlay)

  print("Retail Heatmap Overlay Saved!")

Retail Heatmap Overlay Saved!
